# Faza testowa - zestawienia przekrojowe

Czyta gotowe katalogi przebiegów z `results/runs/` i gotowe pomiary z `results/measurements/`, i pokazuje, ile wnosi każdy komponent w potoku pełnym, jak często w ogóle się odzywa, na ile wynik zależy od doboru wag i czy niekompletność adnotacji zmienia uporządkowanie wkładów. Nie liczy nic na nagraniach i nie ładuje żadnego modelu. Bloki bez przebiegu albo bez pomiaru wypisują, czego brakuje. Tabele poszczególnych eksperymentów są w `test_results.ipynb`.

**Wymaga:** przebiegów i pomiarów, kolejno:

```powershell
python scripts/run_experiment.py configs/full_office.yaml --split test      # oraz tbbt, vatex
python scripts/run_experiment.py configs/full_no_objects_office.yaml --split test
python scripts/run_experiment.py configs/full_swap_caption_office.yaml --split test
python scripts/weight_sensitivity.py --split test
python scripts/measure_cost.py --dataset office --split test
```

Kontrola kompletności ma własną przerwę na ocenę ręczną, opisaną w sekcji 7.

Recall@K w procentach, różnice w punktach procentowych, przecinek dziesiętny, nawias kwadratowy to 95% przedział ufności. Trzy wielkości wkładu wracają w kilku tabelach: Δ_ind to co sygnał dodaje do samej BAZY, Δ_kr to co dodaje do potoku pełnego przy pozostałych sygnałach obecnych, Δ_pod to co się zmienia, gdy jego mechanizm podmienić na drugi.

**Zapisuje:**

| plik | co zawiera |
|---|---|
| `results/figures/weight_sensitivity_<zbior>.png` | krzywe Recall@10 po siatce wag, osobny rysunek na zbiór |
| `results/figures/contribution_cost.png` | wkład wobec kosztu ekstrakcji, punkt na sygnał |
| `<THESIS_FIGURES>/wrazliwosc_wag_<zbior>.png`, `<THESIS_FIGURES>/wklad_koszt.png` | te same rysunki pod nazwami używanymi przez źródła `.tex`, tylko gdy `THESIS_FIGURES` jest ustawione |
| `data/interim/<zbiór>/work/<zbiór>_pool_judgments_new.csv` | pula fragmentów do oceny ręcznej, z pustą kolumną `relevant` |

**Dalej:** `test_appendix.ipynb` - wszystkie miary wszystkich konfiguracji i tabela przejść.

In [ ]:
import importlib
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import compare, figures, pool, sensitivity, tables
from src.evaluation import runs as runs_module
from src.utils import experiments as exp
from src.utils import frozen as frozen_module
from src.utils.vocabulary import COMPLEXITY_VALUES, REQUIREMENT_TAGS

for module in (compare, figures, pool, sensitivity, runs_module, tables, exp,
               frozen_module):
    importlib.reload(module)      # the kernel keeps a once-imported module in memory

SPLIT = "test"
METRIC = "recall@10"

DATASETS = list(exp.DATASETS)
LABEL = exp.DATASET_NAMES
BASE, FULL = exp.BASE, exp.FULL
SIGNALS = list(exp.ADDED_SIGNALS)
SIGNAL_NAME = exp.SIGNAL_NAMES

MEASUREMENTS = ROOT / "results" / "measurements"

# The figure directory of the thesis repository, when there is one. A copy
# lands there under the name the .tex sources use (Polish), while the file in
# results/figures/ keeps the name of this repository (English).
THESIS_FIGURES = None        # e.g. Path(r"C:\\Users\\PC\\...\\praca magisterska\\rysunki")

FROZEN = frozen_module.load_frozen()
runs = runs_module.load_runs(SPLIT)


def available(labels):
    """Runs of the given labels, on the datasets ALL of them cover."""
    present, shared, skipped = runs_module.available(runs, labels)
    for label, reason in skipped:
        print(f"  {label}: {reason}")
    return runs_module.select(runs, present, shared), shared


def missing(labels):
    """Prints what has to be run for a block to have anything to show."""
    gaps = [label for label in labels if label not in runs]
    if gaps:
        print(f"brak przebiegow: {', '.join(gaps)} - patrz naglowek notatnika")
    return bool(gaps)


def winner_label(signal):
    """The variant of a signal the full pipeline carries, or None.

    Delta_ind is "the base plus this signal against the base", and the signal
    the full pipeline holds is the one the development phase chose -- so the
    label comes from the verdict in configs/frozen.yaml, not from a second copy
    of it here. A signal with only one variant has no verdict to read.
    """
    candidates = [label for label, added in exp.ADDS_SIGNAL.items() if added == signal]
    if len(candidates) == 1:
        return candidates[0]
    decision = getattr(FROZEN, signal, None)
    if decision is None:
        return None
    return next((label for label in candidates
                 if exp.LABEL_MECHANISM.get(label) == decision.chosen), None)


def newest(name):
    """The newest measurement of one kind, or None."""
    found = sorted(MEASUREMENTS.glob(f"{name}_*.json"), reverse=True)
    if not found:
        return None
    return found[0], json.loads(found[0].read_text(encoding="utf-8"))["data"]


def difference(pair, desc_ids=None):
    """One contribution as a comparison of two configurations, or None.

    ``pair`` is ``(candidate, reference)`` from experiments.py. Nothing is
    computed here: the pairs are decisions of chapter 6 and the arithmetic is
    compare.compare_variant.
    """
    candidate, reference = pair
    group, _ = available([candidate, reference])
    if len(group) < 2:
        return None
    return compare.compare_variant(group[candidate], group[reference], METRIC,
                                   desc_ids)


print(f"przebiegi znalezione dla czesci {SPLIT!r}: {len(runs)} etykiet")
for label in sorted(runs):
    print(f"  {label:<24}{', '.join(sorted(runs[label]))}")
if not runs:
    print("  (jeszcze zadnego)")
print("\nwerdykty z configs/frozen.yaml:")
for signal in SIGNALS:
    print(f"  {signal:<14}{winner_label(signal) or '?'}")


## 1. Wkłady zbiorczo

Trzy wielkości na sygnał: Δ_ind (BAZA z sygnałem wobec samej BAZY), Δ_kr (potok pełny wobec potoku bez tego sygnału) i Δ_pod (potok pełny wobec potoku z drugim mechanizmem tego sygnału). Wszystkie trzy to zwykłe porównania dwóch konfiguracji: pary etykiet pochodzą z `src/utils/experiments.py`, a różnice z `compare.py`.

Δ_pod istnieje tylko dla sygnałów mających dwa mechanizmy (opisy, obiekty, twarze). Sygnał nieobecny na zbiorze nie ma wiersza z zerem, tylko `-`.

In [ ]:
rows = []
for signal in SIGNALS:
    rows.append(f"{SIGNAL_NAME[signal]} ({', '.join(LABEL[d] for d in exp.datasets_of(signal))})")
    label = winner_label(signal)
    kinds = [("Delta_ind", exp.individual_pair(label) if label else None),
             ("Delta_kr", exp.marginal_pair(signal))]
    if signal in exp.SWAPPED_SIGNALS:
        kinds.append(("Delta_pod", exp.swap_pair(signal)))
    for title, pair in kinds:
        marks = ["n/d" if d not in exp.datasets_of(signal) else "-"
                 for d in DATASETS] + ["-"]
        if pair is None:
            rows.append([title, *marks])
            continue
        result = difference(pair)
        if result is None:
            rows.append([f"{title}  ({pair[0]} - {pair[1]})", *marks])
            continue
        where = exp.datasets_of(signal)
        rows.append([f"{title}  ({pair[0]} - {pair[1]})"]
                    + ["n/d" if d not in where
                       else tables.points(result["per_dataset"][d]["mean"])
                       if d in result["per_dataset"] else "-" for d in DATASETS]
                    + [tables.interval(result["pooled"])])

tables.show("Wklady komponentow",
            ["Wielkosc"] + [LABEL[d] for d in DATASETS] + ["Polaczone [95% PU]"],
            rows, align="l" + "r" * len(DATASETS) + "l",
            note="Delta_ind: BAZA + sygnal wobec BAZY. Delta_kr: potok pelny wobec "
                 "potoku bez tego sygnalu. Delta_pod: potok pelny wobec potoku z "
                 "drugim mechanizmem. Kolumna polaczona dotyczy wylacznie seriali - "
                 "klipy VATEX-a nie wchodza do puli odcinkow. 'n/d' znaczy, ze ten "
                 "sygnal nie jest na tym zbiorze mierzony wcale (decyzja rozdzialu 4); "
                 "myslnik - ze przebiegu jeszcze nie ma.")


## 1a. Różnica wkładu między serialami

Różnica wkładu tego samego sygnału między dwoma serialami, w kolejności *The Big Bang Theory* minus *The Office*. Wkładem odcinka jest różnica Recall@10 między BAZĄ z sygnałem a samą BAZĄ, liczona na zapytaniach tego odcinka - ta sama wielkość, na której opierają się pozostałe różnice.

Przedział pochodzi z `compare.two_sample_interval`: dwie niezależne próby odcinków, wariancja zbiorcza, `df = n1 + n2 - 2`. Rozdział 4 zapisuje błąd standardowy w postaci z osobnymi wariancjami; przy równej liczbie odcinków obie postacie dają identyczny wynik i blok sprawdza to liczbowo dla pierwszego policzonego sygnału.

In [ ]:
import math

PB3_SERIES = ("tbbt", "office")   # the difference is TBBT minus The Office


def series_contributions(pair):
    """Per-episode contributions of one signal, per series."""
    candidate, reference = pair
    group, _ = available([candidate, reference])
    if len(group) < 2:
        return {}
    found = {}
    for dataset in PB3_SERIES:
        left = group[candidate].get(dataset)
        right = group[reference].get(dataset)
        if left is None or right is None:
            continue
        found[dataset] = compare.paired_differences(
            left, right, METRIC, None, compare.inference_unit(dataset))
    return found


def standard_errors(first, second):
    """Standard error of the difference, pooled and with separate variances."""
    na, nb = len(first), len(second)
    mean_a, mean_b = sum(first) / na, sum(second) / nb
    var_a = sum((x - mean_a) ** 2 for x in first) / (na - 1)
    var_b = sum((x - mean_b) ** 2 for x in second) / (nb - 1)
    pooled = ((na - 1) * var_a + (nb - 1) * var_b) / (na + nb - 2)
    return math.sqrt(pooled * (1 / na + 1 / nb)), math.sqrt(var_a / na + var_b / nb)


rows, verified = [], False
for signal in SIGNALS:
    label = winner_label(signal)
    pair = exp.individual_pair(label) if label else None
    per_series = series_contributions(pair) if pair else {}
    if any(dataset not in per_series or len(per_series[dataset]) < 2
           for dataset in PB3_SERIES):
        rows.append([SIGNAL_NAME[signal], "-", "-", "-", "-"])
        continue
    first, second = (per_series[dataset] for dataset in PB3_SERIES)
    entry = compare.two_sample_interval(first, second)
    rows.append([SIGNAL_NAME[signal],
                 f"{len(first)} / {len(second)}",
                 tables.points(sum(first) / len(first)),
                 tables.points(sum(second) / len(second)),
                 tables.interval(entry)])

    if not verified:
        verified = True
        se_pooled, se_separate = standard_errors(first, second)
        equal = len(first) == len(second)
        print(f"kontrola wzoru na bledzie standardowym ({SIGNAL_NAME[signal]}):")
        print(f"  odcinkow: {len(first)} i {len(second)}"
              f"  ({'rowne' if equal else 'ROZNE'})")
        print(f"  wariancja zbiorcza (compare.py): {se_pooled:.12f}")
        print(f"  wariancje osobne (rozdzial 4):   {se_separate:.12f}")
        print("  wzory zgodne" if abs(se_pooled - se_separate) < 1e-12 else
              "  WZORY SIE ROZJEZDZAJA - zglos to zamiast wpisywac liczbe do pracy")
        print()

tables.show("Roznica wkladu sygnalu miedzy serialami, TBBT minus The Office",
            ["Sygnal", "Odcinkow TBBT / Office", "Srednia TBBT", "Srednia Office",
             "Roznica [95% PU]"],
            rows, align="lrrrl",
            note="Wkladem odcinka jest roznica Recall@10 miedzy BAZA z sygnalem "
                 "a sama BAZA, na zapytaniach tego odcinka. Przedzial dwuprobkowy "
                 "o wspolnej wariancji, df = n1 + n2 - 2: odcinki obu seriali nie "
                 "tworza par. Myslnik znaczy, ze ktoregos z dwoch przebiegow nie ma "
                 "na obu serialach.")


## 2. Aktywacja sygnałów

Odsetek zapytań, dla których sygnał w ogóle miał co powiedzieć. Sygnał bramkowany, który odzywa się do co dziesiątego zapytania, nie ruszy średniej kolekcji daleko, choćby odzywał się trafnie. Kolumna **wszystkie** liczy po całym zbiorze zapytań, kolumna **docelowe** po zapytaniach ze znacznikiem przypisanym eksperymentowi tego sygnału (`vocabulary.TAG_EXPERIMENT`, złożone przez `experiments.target_tags`).

E4-D ma osobny wiersz i osobne źródło: nie jest sygnałem, więc nie ma wiersza w `signals.npz`, a jego aktywacja to pole `query_time_detection.active` w `per_query.jsonl`.

In [ ]:
if not missing([FULL]):
    rows = []
    for signal in SIGNALS:
        targets = exp.target_tags(signal)
        for dataset in exp.datasets_of(signal):
            run = runs.get(FULL, {}).get(dataset)
            if run is None:
                rows.append([SIGNAL_NAME[signal], LABEL[dataset], "-", "-", "-"])
                continue
            everywhere = compare.activation_share(run).get(signal)
            wanted = set()
            for tag in targets:
                try:
                    inside, _ = compare.subset_ids(dataset, SPLIT, f"requirements:{tag}")
                except (FileNotFoundError, ValueError):
                    continue
                wanted |= inside
            targeted = (compare.activation_share(run, wanted).get(signal)
                        if wanted else None)
            rows.append([SIGNAL_NAME[signal], LABEL[dataset],
                         ", ".join(targets) or "-",
                         tables.percent(everywhere),
                         tables.percent(targeted) if wanted else "-"])

    # A rejected variant has no BASE + signal run in the FULL pipeline, but the
    # swap pipeline puts it exactly where the frozen one sits, and its signals.npz
    # carries the same two quantities as the rows above. Without these rows the
    # table would say how far the open catalogue reaches only for the variant
    # that lost, which is the comparison the swap effect rests on.
    for signal in exp.SWAPPED_SIGNALS:
        decision = getattr(FROZEN, signal, None)
        rejected = next((label for label, mechanism in exp.LABEL_MECHANISM.items()
                         if exp.ADDS_SIGNAL.get(label) == signal
                         and decision is not None
                         and mechanism == decision.rejected), None)
        if rejected is None:
            continue
        targets = exp.target_tags(signal)
        for dataset in exp.datasets_of(signal):
            run = runs.get(exp.swap_label(signal), {}).get(dataset)
            if run is None:
                continue
            everywhere = compare.activation_share(run).get(signal)
            wanted = set()
            for tag in targets:
                try:
                    inside, _ = compare.subset_ids(dataset, SPLIT, f"requirements:{tag}")
                except (FileNotFoundError, ValueError):
                    continue
                wanted |= inside
            targeted = (compare.activation_share(run, wanted).get(signal)
                        if wanted else None)
            rows.append([f"{SIGNAL_NAME[signal]} ({rejected}, odrzucony)",
                         LABEL[dataset], ", ".join(targets) or "-",
                         tables.percent(everywhere),
                         tables.percent(targeted) if wanted else "-"])

    for dataset in DATASETS:
        run = runs.get("E4-D", {}).get(dataset)
        if run is None:
            continue
        records = [r.get("query_time_detection") for r in run["per_query"]]
        active = [r["active"] for r in records if r is not None]
        rows.append(["E4-D (etap punktacji)", LABEL[dataset], "wymaga_obiektu",
                     tables.percent(sum(active) / len(active) if active else None),
                     "-"])

    tables.show("Zapytania z wlaczonym sygnalem",
                ["Sygnal", "Zbior", "Znacznik docelowy", "wszystkie [%]",
                 "docelowe [%]"], rows, align="lllrr",
                note="Sygnal bramkowany jest wylaczany z wazenia zapytania, dla "
                     "ktorego nie ma nic do powiedzenia - dlatego zasieg jest polowa "
                     "tego, co znaczy wklad. Myslnik w kolumnie 'docelowe' znaczy, ze "
                     "ten zbior nie przypisuje zadnego znacznika tego sygnalu. Wariant odrzucony "
                     "jest mierzony w potoku podmiany, w ktorym zastepuje zamrozony.")


### 2a. Kontrola bramki aktywacji

Sygnał wyłączony z ważenia zapytania ma w nim niczego nie zmieniać: ranking ma być identyczny z rankingiem BAZY, a przejść $0\to1$ i $1\to0$ ma w tej grupie nie być wcale. Nie widać tego z kodu punktacji, bo waga zależy od liczby aktywnych sygnałów i wyłączenie jednego przelicza wagi pozostałych, dlatego blok sprawdza to na wynikach.

Kolumna "identyczny ranking" jest silniejszą połową twierdzenia: nie tylko trafienie w pierwszej dziesiątce się nie zmieniło, lecz cała zapisana lista. Niezerowe `0->1` albo `1->0` znaczy, że założenie jest fałszywe, a nie że przebieg jest zły.

In [ ]:
# a gated signal drops out of the weighting of a query it has nothing to say
# about -- and the claim of chapter 6 is that the query is then scored exactly
# as the base scored it. The weights of the REMAINING signals are recomputed
# when one drops out, so this does not follow from the formula and is checked.
rows = []
for signal in SIGNALS:
    label = winner_label(signal)
    if label is None:
        continue
    for dataset in exp.datasets_of(signal):
        run = runs.get(label, {}).get(dataset)
        base = runs.get(BASE, {}).get(dataset)
        if run is None or base is None:
            continue
        every = {int(record["desc_id"]) for record in run["per_query"]}
        inactive = every - compare.active_ids(run, signal)
        if not inactive:
            rows.append([SIGNAL_NAME[signal], label, LABEL[dataset], 0,
                         "-", "-", "-", "sygnal aktywny dla kazdego zapytania"])
            continue
        counts = compare.transitions(run, base, inactive)
        ranking = {int(r["desc_id"]): [hit["fragment"] for hit in r["top"]]
                   for r in run["per_query"]}
        reference = {int(r["desc_id"]): [hit["fragment"] for hit in r["top"]]
                     for r in base["per_query"]}
        same = sum(ranking[desc_id] == reference[desc_id] for desc_id in inactive)
        rows.append([SIGNAL_NAME[signal], label, LABEL[dataset], len(inactive),
                     counts["0->1"], counts["1->0"], f"{same}/{len(inactive)}",
                     "zgodne" if not (counts["0->1"] or counts["1->0"])
                     and same == len(inactive) else "NIEZGODNE - popraw rozdzial 6"])

if rows:
    tables.show("Kontrola bramki: zapytania, dla ktorych sygnal sie nie wlaczyl",
                ["Sygnal", "Wariant", "Zbior", "Nieaktywnych", "0->1", "1->0",
                 "Identyczny ranking", "Werdykt"], rows, align="lll" + "r" * 4 + "l",
                note="Porownanie z BAZA na zapytaniach, przy ktorych sygnal milczal. "
                     "'Identyczny ranking' liczy zapytania, dla ktorych cala zapisana "
                     "lista fragmentow jest ta sama co w BAZIE, nie tylko trafienie "
                     "w pierwszej dziesiatce.")
else:
    print("brak przebiegow do porownania")

### 2b. Sygnał ruchu na zapytaniach, dla których się włączył

Wszystko liczone na przebiegu BAZA z SlowFastem (E2-C) i wyłącznie na zapytaniach, dla których sygnał ruchu się włączył.

Pierwsza tabela: ile sygnał kosztuje na tych zapytaniach i jak często wśród fragmentów, które podbija ($z>3$), jest fragment poprawny. Oba poziomy Recall@10 to średnie po zapytaniach, ale Δ jest średnią różnic po odcinkach, więc nie jest różnicą dwóch poprzedzających ją kolumn.

Druga tabela pokazuje, dlaczego podbija tak mocno. Każdy sygnał jest przed fuzją standaryzowany osobno dla każdego zapytania: od wartości fragmentu odejmuje się średnią po kolekcji i dzieli przez odchylenie standardowe. Prawdopodobieństwo klasy Kinetics wskazanej przez zapytanie jest dla niemal wszystkich fragmentów bliskie zeru, a dla nielicznych wysokie, więc te nieliczne dostają wartości $z$ wielokrotnie większe niż cokolwiek, co osiąga sygnał sceniczny.

In [ ]:
import csv

import numpy as np

LABEL_SF = "E2-C"      # the base plus motion alone: two signals, weights 1/2 each
BOOSTED = 3.0          # what the paragraph calls a boosted fragment: z above this


def relevant_columns(run, fragments):
    """{desc_id: columns of its relevant fragments}, from the run's relevance.csv."""
    column = {fragment: i for i, fragment in enumerate(fragments)}
    found = {}
    with open(Path(run["dir"]) / "relevance.csv", encoding="utf-8-sig") as handle:
        for record in csv.DictReader(handle, delimiter=";"):
            if record["fragment_id"] in column:
                found.setdefault(int(record["desc_id"]), set()).add(
                    column[record["fragment_id"]])
    return found


loss_rows, scale_rows = [], []
for dataset in DATASETS:
    run = runs.get(LABEL_SF, {}).get(dataset)
    base = runs.get(BASE, {}).get(dataset)
    if run is None or base is None:
        continue
    with np.load(Path(run["dir"]) / "signals.npz", allow_pickle=False) as data:
        names = [str(name) for name in data["names"]]
        values = data["values"].astype(np.float32)       # (signal, query, fragment)
        active = np.asarray(data["active"], dtype=bool)  # (signal, query)
        desc_ids = [int(i) for i in data["desc_ids"]]
        fragments = [str(f) for f in data["fragments"]]
    motion_row = names.index("motion")
    scene_row = names.index(exp.BASE_SIGNAL)
    speaks = np.flatnonzero(active[motion_row])
    wanted = {desc_ids[i] for i in speaks}

    # What the signal costs where it speaks. The two LEVELS are means over
    # queries, like every Recall here; the DELTA between them is a difference of
    # two configurations, so it is read at the unit of inference -- per episode
    # for a series -- and is the same quantity the contribution tables print.
    before = compare.query_mean(base["per_query"], METRIC, wanted)
    after = compare.query_mean(run["per_query"], METRIC, wanted)
    diffs = compare.paired_differences(run, base, METRIC, wanted,
                                       compare.inference_unit(dataset))

    # how often the fragments it boosts include the one the query asks for
    relevant = relevant_columns(run, fragments)
    hit = sum(bool(set(np.flatnonzero(values[motion_row, i] > BOOSTED))
                   & relevant.get(desc_ids[i], set()))
              for i in speaks)

    loss_rows.append([LABEL[dataset], len(wanted), tables.percent(before),
                      tables.percent(after),
                      tables.points(sum(diffs) / len(diffs)) if diffs else "-",
                      tables.number(100 * hit / len(wanted), 1)])

    # why it boosts so hard: both scales on the same queries
    motion = values[motion_row][speaks]
    scene = values[scene_row][speaks]
    scale_rows.append([LABEL[dataset], tables.number(float(motion.max()), 1),
                       tables.number(float(scene.max()), 1),
                       tables.number(float(np.percentile(scene, 99)), 2)])

if loss_rows:
    tables.show("Sygnal ruchu na zapytaniach, dla ktorych sie wlaczyl (rozdz. 6)",
                ["Zbior", "Zapytan", "R@10 BAZA", "R@10 BAZA+SF", "Delta [p.p.]",
                 "poprawny wsrod z>3 [%]"], loss_rows, align="l" + "r" * 5,
                note="Oba poziomy R@10 to srednie po zapytaniach; Delta to srednia "
                     "roznic na jednostce wnioskowania zbioru - po odcinkach dla "
                     "seriali, po klipach dla VATEX-a - wiec czyta sie ja tak samo jak "
                     "wklady w tabelach rozdzialu 6. Dlatego Delta nie jest tu roznica "
                     "dwoch poprzednich kolumn: odcinek z trzema takimi zapytaniami "
                     "ma w niej taka sama wage jak odcinek z trzydziestoma. Ostatnia "
                     f"kolumna: odsetek tych zapytan, w ktorych wsrod fragmentow o z > "
                     f"{BOOSTED:g} jest fragment poprawny.")
    tables.show("Skala sygnalu ruchu po standaryzacji (te same zapytania)",
                ["Zbior", "max z ruchu", "max z sceny", "p99 z sceny"], scale_rows,
                align="lrrr",
                note="Wartosci z z signals.npz przebiegu E2-C, zapisane jako float16, "
                     "wiec ostatnia cyfra moze sie roznic o 1.")
else:
    print("brak przebiegow E2-C albo BAZY")

## 3. PB3 - różnica między grupami zapytań

Dla potoku pełnego: średni Recall@10 na zapytaniach niosących dany znacznik i na pozostałych, z przedziałem dwupróbkowym (`compare.two_sample_interval`, wspólna wariancja, $df = n_1 + n_2 - 2$).

Przedział jest tu inny niż wszędzie indziej i to jest istota tego bloku. Pozostałe tabele porównują dwa przebiegi na tych samych zapytaniach, więc różnica jest sparowana; tutaj porównywane są dwie różne grupy zapytań w jednym przebiegu, a przedział sparowany byłby niewłaściwym narzędziem. Przedział jest świadomie nie-Welchowski: praca podaje jedno wspólne $df$.

Grupy dobiera `PB3_SUBSETS`.

In [ ]:
PB3_SUBSETS = [f"requirements:{tag}" for tag in REQUIREMENT_TAGS]

if not missing([FULL]):
    rows = []
    for spec in PB3_SUBSETS:
        for dataset in DATASETS:
            run = runs.get(FULL, {}).get(dataset)
            if run is None:
                continue
            try:
                inside, outside = compare.subset_ids(dataset, SPLIT, spec)
            except (FileNotFoundError, ValueError):
                continue
            if not inside:
                continue          # this dataset does not assign the tag at all
            values = {r["desc_id"]: r[METRIC] for r in run["per_query"]}
            entry = compare.two_sample_interval(
                [values[i] for i in inside if i in values],
                [values[i] for i in outside if i in values])
            rows.append([spec.split(":", 1)[1], LABEL[dataset],
                         entry["n_a"], entry["n_b"],
                         tables.percent(sum(values[i] for i in inside if i in values)
                                        / max(entry["n_a"], 1)),
                         tables.percent(sum(values[i] for i in outside if i in values)
                                        / max(entry["n_b"], 1)),
                         tables.interval(entry)])

    if rows:
        tables.show("PB3: Recall@10 zapytan ze znacznikiem wobec pozostalych, potok pelny",
                    ["Znacznik", "Zbior", "n ze znacznikiem", "n bez", "R@10 ze",
                     "R@10 bez", "Roznica [95% PU]"], rows, align="ll" + "r" * 5,
                    note="Wielkosc liczona na potoku PELNYM, nie na BAZIE, i w obrebie "
                         "jednego zbioru - to co innego niz roznica miedzy serialami "
                         "z sekcji 1a. "
                         "Przedzial dwuprobkowy o wspolnej wariancji, df = n1 + n2 - 2: "
                         "to sa dwie rozne grupy zapytan w jednym przebiegu, nie ten sam "
                         "zbior zapytan dwa razy, wiec przedzial sparowany bylby tu "
                         "zlym narzedziem. Wiersz powstaje tylko dla zbioru, ktory dany "
                         "znacznik przypisuje.")
    else:
        print("brak przebiegu potoku pelnego albo znacznikow - nic do policzenia")


## 4. PB4 - macierz wkładów według wymagania zapytania

Wiersz na sygnał i zbiór, kolumna na znacznik, plus kolumna złożoności. W komórce stoi Δ_ind tego sygnału policzony wyłącznie na zapytaniach danego podzbioru, czyli odpowiedź na pytanie, czy sygnał pomaga tam, gdzie z założenia powinien.

**Δ liczy się na jednostce wnioskowania zbioru**, a nie na zapytaniu. Dla seriali jednostką jest odcinek: różnica powstaje osobno w każdym odcinku i dopiero te różnice się uśrednia (`compare.paired_differences` z `compare.inference_unit`). To ta sama wielkość, którą kontrasty w `test_results.ipynb` pokazują jako `matching_delta`, więc komórka znacznika docelowego zgadza się tam co do cyfry. Różnica dwóch średnich po całej kolekcji ważyłaby odcinek liczbą zapytań podzbioru, które akurat w nim wypadły. Dla VATEX-a jednostką jest klip i oba sposoby dają tę samą liczbę.

Bez przedziałów: podzbiory znacznikowe bywają małe, a macierz przedziałów byłaby dwudziestoma porównaniami naraz. To rozbicie opisowe, nie zestaw testów.

In [ ]:
PB4_COLUMNS = ([(f"requirements:{tag}", tag) for tag in REQUIREMENT_TAGS]
               + [(f"complexity:{value}", f"zlozonosc {value}")
                  for value in COMPLEXITY_VALUES])

rows = []
for signal in SIGNALS:
    label = winner_label(signal)
    if label is None:
        continue
    candidate, reference = exp.individual_pair(label)
    group, columns = available([candidate, reference])
    if len(group) < 2:
        rows.append([SIGNAL_NAME[signal], "-"] + ["-"] * len(PB4_COLUMNS))
        continue
    for dataset in columns:
        cells = [f"{SIGNAL_NAME[signal]} ({label})", LABEL[dataset]]
        unit = compare.inference_unit(dataset)
        for spec, _ in PB4_COLUMNS:
            try:
                inside, _rest = compare.subset_ids(dataset, SPLIT, spec)
            except (FileNotFoundError, ValueError):
                cells.append("-")
                continue
            if not inside:
                cells.append("-")
                continue
            # at the unit of inference, not at the query: for a series the cell
            # is the mean of the PER-EPISODE differences, which is the quantity
            # the contrast tables of test_results.ipynb call matching_delta, so
            # a target cell here and that table agree to the digit. A difference
            # of two collection means would instead weight an episode by how
            # many of the subset's queries happened to fall in it. For VATEX the
            # unit is the clip and the two ways give the same number.
            diffs = compare.paired_differences(
                group[candidate][dataset], group[reference][dataset], METRIC,
                inside, unit)
            cells.append(tables.points(sum(diffs) / len(diffs)) if diffs else "-")
        rows.append(cells)

if rows:
    tables.show("PB4: wklad indywidualny wedlug wymagania zapytania",
                ["Sygnal", "Zbior"] + [described for _, described in PB4_COLUMNS],
                rows, align="ll" + "r" * len(PB4_COLUMNS),
                note="Delta_ind policzony na zapytaniach danego podzbioru, w punktach "
                     "procentowych, na jednostce wnioskowania zbioru: dla seriali to "
                     "srednia roznic po ODCINKACH (ta sama wielkosc, ktora kontrasty "
                     "w test_results.ipynb podaja jako matching_delta), dla VATEX-a "
                     "srednia roznic po klipach. Bez przedzialow - rozbicie opisowe, "
                     "nie zestaw testow. Myslnik znaczy 'ten podzbior jest tu pusty "
                     "albo przebiegu jeszcze nie ma'.")
else:
    print("brak przebiegow indywidualnych - nic do policzenia")

## 5. Wrażliwość na wagi

Potok daje każdemu aktywnemu sygnałowi wagę $1/|A(q)|$; to wybór, a to zestawienie mówi, ile na nim stoi. Dla każdego sygnału jego waga przebiega siatkę $\{0; 0{,}1; \ldots; 1\} \cup \{1/|A|\}$, a reszta $1-w$ dzieli się równo między pozostałe aktywne sygnały.

Kolumna "zakres" to $\max - \min$ Recall@10 po tej siatce, czyli poziom, a więc średnia po zapytaniach. Kolumna "uporz." mówi, czy w każdym punkcie siatki uporządkowanie wkładów krańcowych zostaje takie samo jak przy wagach jednolitych; wkład jest różnicą dwóch konfiguracji, więc dla seriali liczy się go po odcinkach. To samo dotyczy kolumny "różnica na brzegu".

Punkty nieokreślone to $w = 1$ sygnału bramkowanego. Zapytanie, dla którego sygnał jest nieaktywny, dostaje wtedy sumę wag zero, cała jego kolekcja ma wynik zero, a wtedy każdy fragment ma rangę 1 i zapytanie wchodzi do średniej jako trafienie idealne. Punkt zawyża Recall, a nie zeruje go, więc jest wyłączony z krzywej i z zakresu.

Wszystko liczy `scripts/weight_sensitivity.py` i zapisuje przez `save_measurement`; ten blok wyłącznie czyta.

In [ ]:
found = newest("weight_sensitivity")
if found is None:
    grid, controls = {}, {}
    print("brak pomiaru - uruchom scripts/weight_sensitivity.py --split test")
else:
    path, payload = found
    grid, controls = payload["grid"], payload.get("edge_controls", {})
    print(f"pomiar wczytany z: {path.name}  (czesc {payload.get('split')}, "
          f"{payload.get('metric')})")

    rows = []
    for dataset, swept in grid.items():
        # the uniform weight is 1/|A| of THIS collection, so it comes from the
        # number of signal rows the run stored, never from a constant here
        uniform = 1.0 / len(swept) if swept else None
        rows.append(f"{LABEL.get(dataset, dataset)}  "
                    f"({len(swept)} sygnalow, waga jednolita {tables.number(uniform, 3)})")
        for signal, entry in swept.items():
            check = controls.get(dataset, {}).get(signal, {})
            rows.append([SIGNAL_NAME.get(signal, signal),
                         tables.points(entry["range"], sign=False),
                         {True: "tak", False: "nie", None: "-"}[entry["order_kept"]],
                         ", ".join(tables.number(w, 1)
                                   for w in entry["undefined_points"]) or "-",
                         {True: "zgodna", False: "ROZJAZD", None: "-"}[check.get("close")],
                         tables.points(check.get("difference"))])

    tables.show("Wrazliwosc Recall@10 na dobor wag",
                ["Sygnal", "Zakres [p.p.]", "Uporz.", "Punkty nieokreslone",
                 "Kontrola brzegu", "Roznica na brzegu"], rows, align="lrlllr",
                note="Kontrola brzegu: w = 0 wiersza sygnalu ma odtworzyc przebieg "
                     "full_no_<sygnal>, a w = 1 wiersza bazowego - przebieg BAZY. "
                     "Sygnaly sa zapisane w float16, wiec sprawdzana jest bliskosc, "
                     "a rozbieznosc podrozuje razem z werdyktem. Zakres to poziom "
                     "R@10, wiec srednia po zapytaniach; uporzadkowanie i roznica "
                     "na brzegu to roznice miedzy konfiguracjami, wiec dla seriali "
                     "srednie roznic po odcinkach.")


**Zapisuje:** `results/figures/weight_sensitivity_<zbior>.png`, osobny plik na zbiór. Gdy `THESIS_FIGURES` wskazuje katalog rysunków pracy, powstają tam kopie `wrazliwosc_wag_<zbior>.png`.

Osobny rysunek na zbiór, krzywa na sygnał (razem z sygnałem bazowym), oś do 1,0, pionowa linia odniesienia na wadze jednolitej $1/|A|$ wyliczonej z liczby sygnałów tej kolekcji. Trzy panele w jednym rysunku były nieczytelne: każdy zbiór ma własną skalę Recall@10 i po ściśnięciu do jednej trzeciej szerokości krzywe zlewały się ze sobą. Legenda wymienia tylko sygnały mierzone na danym zbiorze; VATEX ma ich cztery. Znacznik jest drugim kodowaniem sygnału obok koloru, żeby wydruk w skali szarości dało się czytać, a paleta jest bezpieczna dla daltonizmu (Okabe-Ito). Skład jak w pracy (`usetex`); bez zainstalowanego LaTeX-a rysunki powstają w krojach matplotliba i blok to wypisze.

**Ten blok potrzebuje wyniku poprzedniego** - czyta `grid` wczytany razem z tabelą powyżej.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

usetex = figures.style()
PCT = figures.percent_sign()

panels = [dataset for dataset in DATASETS if dataset in grid] or list(grid)
if not panels:
    print("brak pomiaru weight_sensitivity - zaden rysunek nie powstal")

for dataset in panels:
    swept = grid[dataset]
    fig, axis = plt.subplots(figsize=(5.8, 4.3))
    drawn = []
    for signal, entry in swept.items():
        if not entry["curve"]:
            continue
        spec = figures.SIGNAL_STYLE.get(signal, {"color": "0.4", "marker": "o"})
        weights = [w for w, _ in entry["curve"]]
        recall = [100 * value for _, value in entry["curve"]]
        axis.plot(weights, recall, color=spec["color"], marker=spec["marker"],
                  markersize=5.5, linewidth=1.6, markeredgecolor="white",
                  markeredgewidth=0.7)
        drawn.append(signal)

    axis.set_title(LABEL.get(dataset, dataset), style="italic")
    axis.set_xlabel(r"waga badanego sygnalu $w_i$")
    axis.set_ylabel(f"Recall@10 [{PCT}]")
    axis.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    axis.set_xlim(-0.03, 1.03)
    axis.margins(y=0.12)

    # the uniform weight is 1/|A| of THIS collection, so it comes from the number
    # of rows this collection was swept over, never from a constant here. The
    # line is drawn after the limits are settled, so that the caption beside it
    # sits on the axis and not above it.
    uniform = 1.0 / len(swept) if swept else None
    if uniform is not None:
        axis.axvline(uniform, color="0.35", ls="--", lw=1.2)
        axis.text(uniform + 0.014, axis.get_ylim()[0], "  waga jednolita",
                  rotation=90, va="bottom", ha="left", fontsize=8.5, color="0.35")

    # only the signals of THIS collection: VATEX is measured on four of them and
    # a legend carrying the two it never had would read as curves that went
    # missing from the panel
    order = [s for s in figures.SIGNAL_STYLE if s in set(drawn)]
    handles = [Line2D([0], [0], color=figures.SIGNAL_STYLE[s]["color"],
                      marker=figures.SIGNAL_STYLE[s]["marker"], lw=1.6, markersize=5.5,
                      markeredgecolor="white", label=figures.figure_label(s))
               for s in order]
    rows = 0
    if handles:
        # at most three entries in a row, and the rows filled evenly: the panel is
        # a third of the width it had while the three collections shared one
        # figure, so a single row of six entries would set the width of the whole
        # thing, and VATEX's four would hang as three plus one
        rows = -(-len(handles) // 3)
        fig.legend(handles=handles, loc="lower center", ncol=-(-len(handles) // rows),
                   frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.05 * rows, 1, 1))
    figures.save(fig, f"weight_sensitivity_{dataset}", THESIS_FIGURES,
                 f"wrazliwosc_wag_{dataset}")

## 6. Wkład wobec kosztu ekstrakcji

**Zapisuje:** `results/figures/contribution_cost.png`, plus kopię w `THESIS_FIGURES`, gdy jest ustawione.

Punkt na sygnał, oś pozioma to koszt ekstrakcji w minutach na godzinę materiału (średnia ze wszystkich `cost_*.json` niosących sekcję ekstrakcji, czyli z obu seriali), oś pionowa to wkład krańcowy w punktach procentowych. Sygnał tani i wnoszący dużo leży w lewym górnym rogu.

**Regiony twarzy i tożsamość płacą oba za bufor wycinków RetinaFace'a.** Bufor powstaje raz, ale żaden z tych dwóch sygnałów bez niego nie działa, więc jego koszt wchodzi do obu punktów; inaczej wypadłoby przypisać go arbitralnie jednemu.

In [ ]:
import matplotlib.pyplot as plt

# which extraction components a signal pays for. Keyed by the mechanism the
# configuration names, because the cost of "the caption signal" is the cost of
# the generator the pipeline actually carries.
COST_COMPONENTS = {
    "caption": {"blip": ["blip"], "llava_1_5_7b": ["llava"]},
    "objects": {"yolo11": ["yolo11"], "yoloe_promptfree": ["yoloe"]},
    "motion": {None: ["slowfast"]},
    "face_regions": {"clip_regions": ["retinaface"],
                     "hsemotion": ["retinaface", "hsemotion"]},
    "identity": {None: ["retinaface", "arcface"]},
}


def mechanism_of(signal):
    """The mechanism the frozen pipeline carries for a signal, or None."""
    decision = getattr(FROZEN, signal, None)
    return getattr(decision, "chosen", None)


def extraction_costs(split=SPLIT):
    """Cost per component in minutes per hour, AVERAGED over the series.

    Same selection rule as the cost table in cost.ipynb: the measurement of
    every series on this split, not the newest file. The cost of a component is
    a rate per hour of material and the two series give two different rates --
    the caption generator most of all, because it writes longer descriptions on
    one of them. Taking the newest file alone would put this axis on ONE series
    while the contribution on the other axis is pooled over BOTH, so the two
    axes would describe different material; and a figure disagreeing with the
    table beside it is worse than either number alone.

    Returns ``({key: min_per_hour}, {key: [datasets]}, [(file, dataset, h)])``.
    """
    per_key, sources = {}, []
    for path in sorted(MEASUREMENTS.glob("cost_*.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))["data"]
        if payload.get("split") != split or payload.get("dataset") not in exp.SERIES:
            continue
        if not payload.get("extraction"):
            continue          # a --only run, or the separate E4-D measurement
        sources.append((path.name, payload["dataset"], payload.get("corpus_hours")))
        for entry in payload["extraction"]:
            per_key.setdefault(entry["key"], []).append(
                (payload["dataset"], entry["min_per_hour"]))
    return ({key: sum(v for _, v in values) / len(values)
             for key, values in per_key.items()},
            {key: sorted(d for d, _ in values) for key, values in per_key.items()},
            sources)


cost, cost_on, cost_sources = extraction_costs()
if not cost:
    print(f"brak pomiaru kosztu na czesci {SPLIT!r} - uruchom scripts/measure_cost.py")
else:
    print(f"koszt usredniony z {len(cost_sources)} pomiarow (czesc {SPLIT!r}):")
    for name, dataset, hours in cost_sources:
        print(f"  {name}  ({LABEL.get(dataset, dataset)}, {hours} h korpusu)")
    # a component measured on one series only is not an average and the point it
    # puts on the figure rests on half the material the other points rest on
    lonely = {key: where for key, where in cost_on.items()
              if len(where) < len(cost_sources)}
    if lonely:
        print("\nUWAGA: te komponenty pochodza z jednego zbioru, wiec NIE sa srednia:")
        for key, where in sorted(lonely.items()):
            print(f"  {key:<22}{', '.join(LABEL.get(d, d) for d in where)}")

points = []
for signal in SIGNALS:
    keys = COST_COMPONENTS.get(signal, {}).get(mechanism_of(signal))
    if keys is None or any(key not in cost for key in keys):
        continue
    minutes = sum(cost[key] for key in keys)
    result = difference(exp.marginal_pair(signal))
    value = result["pooled"]["mean"] if result else None
    if value is None:
        continue
    points.append((signal, minutes, 100 * value))

usetex = figures.style()
PCT = figures.percent_sign()
fig, axis = plt.subplots(figsize=(6.4, 4.6))
if not points:
    axis.text(0.5, 0.5, "brak pomiaru kosztu albo przebiegow full_no_*",
              ha="center", va="center")
    axis.set_axis_off()
else:
    for signal, minutes, value in points:
        spec = figures.SIGNAL_STYLE[signal]
        axis.scatter(minutes, value, color=spec["color"], marker=spec["marker"],
                     s=95, edgecolor="white", linewidth=0.8, zorder=3)
        axis.annotate(figures.figure_label(signal), (minutes, value),
                      textcoords="offset points", xytext=(8, 5), fontsize=9)
    axis.axhline(0, color="0.35", lw=1.0)
    # log scale, as section koszt-obliczeniowy describes it: the cheapest
    # component costs 0,17 min/h and the caption generator 72,5, so on a linear
    # axis five of the six points land on top of one another at zero
    axis.set_xscale("log")
    koszty = [minutes for _, minutes, _ in points]
    axis.set_xlim(min(koszty) / 2.5, max(koszty) * 6)   # room for the last label
    axis.set_xlabel("koszt ekstrakcji [min na godzinę materiału]")
    axis.set_ylabel("wkład krańcowy [p.p. Recall@10]")
    axis.margins(y=0.18)
    figures.save(fig, "contribution_cost", THESIS_FIGURES, "wklad_koszt")

    tables.show("Wklad krancowy wobec kosztu ekstrakcji (fig. wklad-koszt)",
                ["Sygnal", "Koszt [min/h]", "Delta_kr [p.p.]"],
                [[figures.signal_label(s), tables.number(m, 2), tables.points(v / 100)]
                 for s, m, v in points],
                note="Wklad krancowy jest polaczony po odcinkach obu seriali. Bufor "
                     "wycinkow RetinaFace'a wchodzi do kosztu regionow twarzy i "
                     "tozsamosci naraz: powstaje raz, ale zaden z tych sygnalow bez "
                     "niego nie dziala.")


## 7. Kontrola kompletności adnotacji

Adnotacja mówi, który fragment odpowiada na zapytanie. Nie mówi, że żaden inny fragment na nie nie odpowiada, a przy przeszukiwaniu całego odcinka zamiast krótkiego klipu, dla którego adnotację pisano, ta różnica jest realna: system może zwrócić poprawny fragment i dostać za niego zero, bo nikt tego fragmentu nie zapisał.

Kontrola ma trzy kroki i środkowy jest ręczny. Blok niżej wykonuje pierwszy: losuje 10% zapytań testowych na serial (ziarno zapisane z góry), zbiera z ich rankingów fragmenty głównych konfiguracji, odrzuca te już uznane za poprawne, miesza wiersze i zapisuje plik bez kolumny konfiguracji, żeby po odpowiedzi nie dało się poznać, który wariant fragment zwrócił.

VATEX puli nie ma: klipy trwają dziesięć sekund, a adnotacją jest opis samego klipu, więc luka, którą ta kontrola mierzy, nie ma się gdzie otworzyć.

**Zapisuje:** `data/interim/<zbiór>/work/<zbiór>_pool_judgments_new.csv`.

In [ ]:
report = pool.build_pool(SPLIT, runs=runs)

rows = []
for dataset, entry in report.items():
    rows.append([LABEL[dataset], entry["queries"], len(entry["labels"]),
                 entry["rows"], Path(entry["file"]).name if entry["file"] else "-"])
tables.show("Pula ocen: co poszlo do oceny recznej",
            ["Zbior", "Zapytan w probie", "Konfiguracji", "Fragmentow", "Plik"],
            rows, align="lrrrl",
            note=f"Proba to {pool.POOL_SHARE:.0%} zapytan testowych na serial, ziarno "
                 f"{pool.POOL_SEED}; zestaw {len(pool.POOL_LABELS)} konfiguracji jest "
                 "zapisany przed pomiarem. Konfiguracja bez przebiegu jest nazwana "
                 "i pominieta - pula zbudowana na czesci zestawu wciaz jest pula, "
                 "o ile raport to mowi.")


### Przerwa: ocena ręczna

Dalsze bloki nie mają czego liczyć, dopóki plik puli nie zostanie oceniony ręcznie.

1. Otwórz `data/interim/<zbiór>/work/<zbiór>_pool_judgments_new.csv` i wypełnij kolumnę `relevant` wartością `yes` albo `no`: czy ten fragment odpowiada na to zapytanie.
2. Wypełniony plik skopiuj do `data/annotations/<zbiór>/` pod tą samą nazwą bez `_new`. Kolumn nie trzeba przycinać - przeliczenie czyta `desc_id`, `fragment` i `relevant`, a reszta zostaje, bo to ona pozwala wrócić do wiersza.
3. Żaden notatnik ani skrypt do `data/annotations/` nie pisze; ten plik przenosi się ręcznie, tak samo jak zweryfikowane pliki adnotacji (`CLAUDE.md`).

Odpowiedź `no` niczego nie kasuje: fragment odrzucony przez pulę i tak nie był w adnotacji. Odrzucenie fragmentu, który jest w adnotacji, byłoby poprawką adnotacji, czyli innym działaniem, robionym ręcznie i w niej samej.

In [ ]:
rows = []
for dataset in exp.SERIES:
    path = pool.judgments_file(dataset)
    judgments = pool.load_judgments(path)
    if not judgments:
        print(f"{LABEL[dataset]}: brak ocen ({path.name}) - patrz komorka wyzej")
        continue
    for label in pool.POOL_LABELS:
        run = runs.get(label, {}).get(dataset)
        if run is None:
            continue
        result = pool.recompute(run, judgments)
        rows.append([LABEL[dataset], exp.display(label, base_as_name=False),
                     result["n"], tables.percent(result["before"]),
                     tables.percent(result["after"]),
                     tables.points((result["after"] - result["before"])
                                   if result["before"] is not None else None),
                     result["gained"]])

if rows:
    tables.show("Kontrola kompletnosci adnotacji",
                ["Zbior", "Konfiguracja", "Zapytan w puli", "R@10 przed",
                 "R@10 po", "Roznica [p.p.]", "Naprawionych"], rows,
                align="ll" + "r" * 5,
                note="Przeliczenie obejmuje wylacznie zapytania z puli: zapytanie spoza "
                     "niej ma adnotacje, ktora mialo zawsze, wiec mieszanie ich "
                     "usredniloby miare uzupelniona z nieuzupelniona i nie podalo "
                     "zadnej z nich. Wynikiem kontroli nie jest lepszy Recall, tylko "
                     "wielkosc luki - i to, czy uporzadkowanie wkladow ja przetrwalo.")
else:
    print("brak ocen recznych albo przebiegow - nic do przeliczenia")
